# Centralized Overvoltage Hosting Capacity Example

This notebook calculates the **overvoltage hosting capacity for each bus** in the feeder (centralized allocation) and visualizes the results on the circuit diagram.

In [1]:
#!pip install py-dss-interface
#!pip install py-dss-toolkit
#!git clone https://github.com/PauloRadatz/opendss-python-examples.git
#%cd opendss-python-examples

In [2]:
import pathlib
import numpy as np
import pandas as pd
import py_dss_interface
from py_dss_toolkit import dss_tools

# --- Helper functions ---
STEP_KW = 100
MAX_KW = 10000
OV_THRESHOLD = 1.05

def add_gen(dss, gen_bus, gen_kv):
    for gen in gen_bus:
        dss.text(f"new generator.{gen} phases=3 bus1={gen_bus[gen]} kv={gen_kv[gen]} "
                 f"kw=0.0001 pf=1 Vminpu=0.7 Vmaxpu=1.2")

def increase_gen(dss, gen_kw):
    for gen, kw in gen_kw.items():
        dss.text(f"Edit generator.{gen} kw={kw}")

def solve_powerflow(dss):
    dss.text("solve")

def check_overvoltage_violation(dss):
    voltages = dss.circuit.buses_vmag_pu
    return max(voltages) > OV_THRESHOLD

def set_load_level_condition(dss, load_mult):
    dss.text(f"set loadmult={load_mult}")

def result_centralized_info(bus, metric, hc_value, load_condition, existing_gen, device_type):
    return f"""Summary of the Centralized Hosting Capacity Analysis

Feeder Conditions:
    Load level condition = {load_condition}
    Existing Gen considered = {existing_gen}

Hosting Capacity:
    Bus = {bus}
    Value = {hc_value} kW
    Device Type = {device_type}
    Metric = {metric}"""

## Define feeder and create OpenDSS object and connect to dss_tools

In [3]:
start_dir = pathlib.Path.cwd().resolve()
repo_root = next(
    parent for parent in [start_dir, *start_dir.parents]
    if (parent / "feeder_models").exists()
)
dss_file = repo_root / "feeder_models" / "8bus" / "Master.dss"
dss = py_dss_interface.DSS()

dss_tools.update_dss(dss)

print(f"Using feeder file: {dss_file}")

Using feeder file: C:\PauloRadatz\GitHub\opendss-python-examples\feeder_models\8bus\Master.dss


## Calculate hosting capacity for each bus

In [4]:
load_mult = 0.2
buses = ["1", "2", "3", "4", "5", "6", "7"]
hc_bus_dict = {}

for bus in buses:
    # Compile OpenDSS feeder model (fresh circuit for each bus)
    dss.text(f'compile "{dss_file}"')
    set_load_level_condition(dss, load_mult=load_mult)

    # Select bus and get base voltage
    dss.circuit.set_active_bus(bus)
    kv = dss.bus.kv_base * np.sqrt(3)

    gen_bus = {"gen": dss.bus.name}
    gen_kv = {"gen": kv}

    # Add generator at bus
    add_gen(dss, gen_bus, gen_kv)

    # Hosting capacity loop
    hosting_capacity_value = MAX_KW
    i = 0
    while i * STEP_KW < MAX_KW:
        i += 1
        i_kw = i * STEP_KW
        gen_kw = {"gen": i_kw}

        increase_gen(dss, gen_kw)
        solve_powerflow(dss)

        if check_overvoltage_violation(dss):
            hosting_capacity_value = (i - 1) * STEP_KW
            break

    hc_bus_dict[bus] = hosting_capacity_value / 1000.0  # Store in MW
    print(f"Bus {bus}: {hosting_capacity_value} kW ({hc_bus_dict[bus]:.2f} MW)")

Bus 1: 6000 kW (6.00 MW)
Bus 2: 3100 kW (3.10 MW)
Bus 3: 2300 kW (2.30 MW)
Bus 4: 2100 kW (2.10 MW)
Bus 5: 1200 kW (1.20 MW)
Bus 6: 1000 kW (1.00 MW)
Bus 7: 900 kW (0.90 MW)


## Display the results

In [5]:
results_df = pd.DataFrame(
    list(hc_bus_dict.items()),
    columns=['Bus', 'Overvoltage Hosting Capacity (MW)']
)
results_df

,Bus,Overvoltage Hosting Capacity (MW)
0,1,6.0
1,2,3.1
2,3,2.3
3,4,2.1
4,5,1.2
5,6,1.0
6,7,0.9


In [6]:
bus_to_feeder = {}
for elem_name in dss.circuit.elements_names:
    cls = elem_name.split(".")[0].lower()
    if cls not in ("line", "transformer"):
        continue
    dss.circuit.set_active_element(elem_name)
    bus_names = dss.cktelement.bus_names
    if len(bus_names) >= 2:
        to_bus = bus_names[1].split(".")[0]
        bus_to_feeder[to_bus] = elem_name.lower()

results_by_line = [
    (bus_to_feeder.get(bus, bus), hc_mw)
    for bus, hc_mw in hc_bus_dict.items()
]
results_df = pd.DataFrame(
    results_by_line,
    columns=["Feeding element", "Overvoltage Hosting Capacity (MW)"]
).set_index("Feeding element")
results_df

,Overvoltage Hosting Capacity (MW)
Feeding element,
line.l01,6.0
line.l12,3.1
line.l23,2.3
line.l24,2.1
transformer.t45,1.2
line.l56,1.0
line.l57,0.9


## Plot hosting capacity on the circuit diagram

In [7]:
sub_marker = dss_tools.interactive_view.circuit_get_bus_marker("0", "square", 12, "black")
gen_marker = dss_tools.interactive_view.circuit_get_bus_marker("4", "circle", 10, "red")

dss_tools.interactive_view.user_numerical_defined_settings.results = results_df['Overvoltage Hosting Capacity (MW)']
dss_tools.interactive_view.user_numerical_defined_settings.unit = "kW"
dss_tools.interactive_view.user_numerical_defined_settings.colorbar_title = "HC in kW"
dss_tools.interactive_view.circuit_plot(parameter="user numerical defined", title="Centralized Hosting Capacity")

## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [pauloradatz.me](https://www.pauloradatz.me)
- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)